# Minimal HITL Simulation

This notebook builds a small synthetic human-in-the-loop (HITL) simulation to illustrate a simple point:

> Observed acceptance is not the same thing as true system quality.

A model can improve while the observed metric stays noisy because the metric is filtered through user behavior.


## Formulation

Each user is represented by a 2x2 response matrix:

```text
H = [
  [p_ac,       p_ai      ],
  [1 - p_ac,   1 - p_ai  ]
]
```

Where:

- `p_ac` is the probability of accepting a correct output.
- `p_ai` is the probability of accepting an incorrect output.

The system is represented by a state vector:

```text
p_s = [
  [p_s,c      ],
  [1 - p_s,c  ]
]
```

Where `p_s,c` is the probability that the system output is correct.

The observed outcome is:

```text
p_o = H @ p_s
```

So the observed acceptance probability is:

```text
P(accept) = p_ac * p_s,c + p_ai * (1 - p_s,c)
```

That acceptance rate is a proxy metric, not a direct readout of model correctness.


In [15]:
import math

import numpy as np
import pandas as pd
import plotly.express as px

## A Single User Against A Static System

Start with one user and one fixed system quality value. This makes the matrix interpretation concrete before we scale up to a population and a time series.


In [16]:
ps_c = 0.95
p_ac = 0.80
p_ai = 0.00

ps = np.array([[ps_c], [1 - ps_c]])
H = np.array([[p_ac, p_ai], [1 - p_ac, 1 - p_ai]])
observed_outcome = H @ ps

pd.DataFrame(
    {
        "Outcome": ["Accepted", "Rejected"],
        "Probability": observed_outcome.flatten(),
    }
)

,Outcome,Probability
0,Accepted,0.76
1,Rejected,0.24


Even with a strong system (`95%` correct), the observed acceptance rate drops to `76%` here because the human reviewer rejects some correct outputs.


## Build A User Population

Next we create a population of users with slightly different review behavior. This is the source of much of the variance in the observed metric.


In [17]:
rng = np.random.default_rng(seed=7)

n_users = 100


def build_human_matrix(p_ac, p_ai):
    return np.array([[p_ac, p_ai], [1 - p_ac, 1 - p_ai]])


user_ensemble = {}
user_profiles = []

for i in range(n_users):
    user_id = f"u{i + 1}"
    p_ac = rng.uniform(low=0.8, high=1.0)
    p_ai = rng.uniform(low=0.0, high=0.2)
    user_ensemble[user_id] = build_human_matrix(p_ac, p_ai)
    user_profiles.append({"User": user_id, "p_ac": p_ac, "p_ai": p_ai})

user_profiles_df = pd.DataFrame(user_profiles)
user_profiles_df.head(10)

,User,p_ac,p_ai
0,u1,0.925019,0.179443
1,u2,0.955137,0.045041
2,u3,0.860033,0.174711
3,u4,0.801053,0.164246
4,u5,0.959414,0.093587
5,u6,0.860606,0.055685
6,u7,0.850974,0.089015
7,u8,0.900910,0.110699
8,u9,0.999100,0.158532
9,u10,0.924436,0.197792


## Simulation Helpers

We now define:

- a daily traffic simulation, where requests are randomly assigned to users
- a system-quality curve, which represents steady model improvement over time


In [18]:
def simulate_daily_activity(day, system_correct_prob, n_items, user_ensemble, rng):
    ps = np.array([[system_correct_prob], [1 - system_correct_prob]])
    telemetry = []
    user_ids = list(user_ensemble.keys())

    for _ in range(n_items):
        user = rng.choice(user_ids)
        accept_prob = float((user_ensemble[user] @ ps)[0, 0])
        is_accepted = rng.random() < accept_prob
        telemetry.append(
            [day, user, is_accepted, system_correct_prob, accept_prob]
        )

    return pd.DataFrame(
        telemetry,
        columns=[
            "Date",
            "User",
            "isAccepted",
            "hiddenSystemState",
            "acceptProbability",
        ],
    )


def system_quality_over_time(day):
    development_time = 365
    baseline = 0.3
    initial_gains = (day / development_time) * math.exp(
        -((2 * day / development_time) ** 2)
    )
    diminishing_returns = math.log(day / development_time + 1)
    return baseline + initial_gains + diminishing_returns

## Run The Simulation

This produces a synthetic telemetry table for 455 days, with 1,000 requests per day.


In [19]:
n_items_per_day = 1000
simulation_duration = 365 + 90

historical_telemetry = []

for day_idx in range(simulation_duration):
    system_correct_prob = min(system_quality_over_time(day_idx), 1.0)
    historical_telemetry.append(
        simulate_daily_activity(
            day=day_idx + 1,
            system_correct_prob=system_correct_prob,
            n_items=n_items_per_day,
            user_ensemble=user_ensemble,
            rng=rng,
        )
    )

historical_telemetry_df = pd.concat(historical_telemetry, ignore_index=True)
historical_telemetry_df.head()

,Date,User,isAccepted,hiddenSystemState,acceptProbability
0,1,u65,False,0.3,0.299213
1,1,u82,False,0.3,0.306147
2,1,u24,False,0.3,0.321250
3,1,u20,True,0.3,0.311427
4,1,u24,False,0.3,0.321250


The `hiddenSystemState` column exists only because this is a simulation. In a real production setting, that latent quality value would not be directly observable.


## Aggregate Daily Metrics

To study the proxy gap, compare the observed acceptance rate against the hidden true system quality.


In [6]:
daily_metrics = (
    historical_telemetry_df.groupby("Date")
    .agg(
        observed_accept_rate=("isAccepted", "mean"),
        hidden_system_quality=("hiddenSystemState", "mean"),
        avg_accept_probability=("acceptProbability", "mean"),
        sampled_users=("User", "nunique"),
    )
    .reset_index()
)

daily_metrics.head()

,Date,observed_accept_rate,hidden_system_quality,avg_accept_probability,sampled_users
0,1,0.334,0.300000,0.341540,100
1,2,0.336,0.305476,0.347618,100
2,3,0.377,0.310943,0.349427,100
3,4,0.367,0.316403,0.355027,100
4,5,0.346,0.321853,0.359919,100


In [ ]:
comparison_df = daily_metrics.melt(
    id_vars="Date",
    value_vars=["hidden_system_quality", "observed_accept_rate"],
    var_name="Metric",
    value_name="Rate",
)

comparison_df["Metric"] = comparison_df["Metric"].map(
    {
        "hidden_system_quality": "Hidden true system quality",
        "observed_accept_rate": "Observed acceptance rate",
    }
)

fig = px.scatter(
    comparison_df,
    x="Date",
    y="Rate",
    color="Metric",
    title="Observed Acceptance vs Hidden System Quality",
)
fig.update_layout(yaxis_tickformat=".0%", legend_title_text="")
fig.show()

This chart is the central takeaway. The system improves monotonically in the simulation, but the observed acceptance rate is a transformed and noisier signal.


## User-Level Noise

The next plot shows daily acceptance rate by user. The horizontal bands at values like `0`, `1/2`, `1/3`, and `1` come from small per-user sample sizes on a given day, which makes the telemetry look much noisier than the hidden system state.


In [8]:
observed_daily_accepts = (
    historical_telemetry_df.groupby(["Date", "User"])["isAccepted"]
    .mean()
    .reset_index()
)

fig = px.scatter(
    observed_daily_accepts,
    x="Date",
    y="isAccepted",
    color="User",
    hover_name="User",
    opacity=0.45,
    title="Daily Acceptance Rate by User",
    labels={"isAccepted": "Acceptance Rate", "Date": "Day"},
)
fig.update_traces(marker={"size": 5})
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

## Takeaways

- Acceptance is a proxy metric shaped by both model quality and reviewer behavior.
- Heterogeneous users create variance even when the underlying system trend is smooth.
- Sparse user-day samples introduce visible quantization artifacts in the telemetry.
- A model can improve while the observable metric remains noisy or biased.

That is the core long-tail evaluation lesson this notebook is designed to highlight.
